In [ ]:
"""
sandbox_time.ipynb

A sandbox to develop a time-resolved class.

Author: Stellina X. Ao
Created: 2026-07-07
Last Modified: 2026-07-07
Python Version: 3.11.14
"""


import scienceplots  # noqa: F401
import shutup
import matplotlib.pyplot as plt

%load_ext autoreload
%autoreload 2

# pretty plots
plt.style.use(["nature"])
plt.rcParams["figure.dpi"] = 200
%matplotlib widget
%config InlineBackend.print_figure_kwargs = {'bbox_inches':None}

# suppress warnings :-)
shutup.please()

In [ ]:
subj_id = "MR82"
sess_id = "20251027_152036"

In [ ]:
"""--------------------------------------------"""
""" finish class """
# beta weight sanity checks
# check weights

# weight renderer
# opt 1: same thing as right now, but expand to a matrix instead
# opt 2: psths and weight traces

# sanity checks

# intercept? normalize? (slacked fig is no intercept, no norm, and only response) one v. two regressors?

# baseline fit should be reconsidered
# add intervals for regressors
"""--------------------------------------------"""

## init

In [ ]:
from sg.models import TimeResolvedEncoder

encoder = TimeResolvedEncoder(
    subj_id,
    sess_id,
    stepsize_s=0.1,
    # tv_keys=["response"],
    norm=False,
)
encoder.baseline_predict()
encoder.encoder_predict()

In [ ]:
encoder.binwidth_ms

In [ ]:
encoder.get_r2()

In [ ]:
encoder.encoder_weights.shape

In [ ]:
encoder.psths["DLS"].shape

In [ ]:
encoder.view_weights(mode="matrix")

In [ ]:
from squiggs.renderers import PETHWeightRendererTime
from squiggs.neuron_viewer import NeuronViewer
from utils.paths import FIGURES_DIR
from core.data import tv_vals, get_psths_cond

reg = "DLS"
model = "encoder"
mode = "response"


# r = PETHWeightRendererTime(
#     weights=encoder.encoder_weights[:,encoder.reg_idxs[reg],:],
#     weight_names=encoder.dm_names,
#     peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
#     binwidth_s=0.1,
#     tbin_centers=encoder.tbin_centers,
# )

r = PETHWeightRendererTime(
    weights=encoder.encoder_weights[:, encoder.reg_idxs[reg], :],
    tv="response",
    weight_idxs=encoder.dm_idxs,
    tv_vals=tv_vals,
    mode="trace",
    peths=get_psths_cond(encoder.psths[reg], encoder.trial_data, mode=mode),
    binwidth_s=0.1,
    tbin_centers=encoder.tbin_centers,
)

# r = WeightRendererTime(
#     weights=encoder.encoder_weights[:,encoder.reg_idxs[reg],:],
#     tv="response",
#     weight_idxs=encoder.dm_idxs,
#     tv_vals=tv_vals,
#     tbin_centers=encoder.tbin_centers,
# )


nv = NeuronViewer(
    num_units=encoder.psths[reg].shape[0], render_func=r, fig_dir=FIGURES_DIR
)

In [ ]:
encoder.tbin_centers

In [ ]:
all([encoder_.robs.min() == 0 for encoder_ in encoder.t_encoders.values()])

In [ ]:
encoder.robs_predict["encoder"]

In [ ]:
encoder.view_fits()

In [ ]:
# plt.figure(tight_layout=True)
# plt.plot(
#     encoder.tbins[:-1],
#     encoder.encoder_weights[:, i, encoder.dm_idxs["response_right"]],
#     color="#A426BA",
# )
# plt.plot(
#     encoder.tbins[:-1],
#     encoder.encoder_weights[:, i, encoder.dm_idxs["response_left"]],
#     color="#1A973D",
# )
# plt.xlabel("Time (s)")
# plt.ylabel("beta weight")
# plt.axvline(x=0, linestyle="--", color="#666666")
# plt.axhline(y=0, color="k", linewidth=0.2)
# plt.show()

# trial-avg

In [ ]:
from sg.models import Encoder, StrategyEncoder

encoder = Encoder(subj_id, sess_id)
encoder.get_r2()
encoder.verify()

encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")
encoder_mf = StrategyEncoder(subj_id, sess_id, strategy_filter="mf")

encoder_mb.verify()
encoder_mb.fit_encoder()

encoder_mf.verify()
encoder_mf.fit_encoder()

In [ ]:
encoder.num_units

In [ ]:
encoder.encoder_weights.shape

In [ ]:
import numpy as np

np.arange(-0.5, 1, 0.12)

In [ ]:
encoder.view_fits()

In [ ]:
encoder.view_weights()

In [ ]:
from sg.models import ShuffledEncoder

se = ShuffledEncoder(
    subj_id,
    sess_id,
    tv_keys=[
        "response",
        "rewarded",
        "block_side",
        "strategy",
        "response_prev",
        "rewarded_prev",
    ],
)
se.plot_cvr2()
se.plot_dr2()
se.plot_bound_r2()

## r2 comp between regions and strategies 

scatter version is in .verify()

In [ ]:
import numpy as np

np.mean(encoder_mb.scores["encoder"] > encoder_mf.scores["encoder"])

In [ ]:
encoder_mb_cond = StrategyEncoder(
    subj_id, sess_id, strategy_filter="mb", balance_strategy=True, cond_balance=True
)
encoder_mb_count = StrategyEncoder(
    subj_id, sess_id, strategy_filter="mb", balance_strategy=True, cond_balance=False
)
encoder_mf_count = StrategyEncoder(
    subj_id, sess_id, strategy_filter="mf", balance_strategy=True, cond_balance=False
)
encoder_mb = StrategyEncoder(subj_id, sess_id, strategy_filter="mb")

In [ ]:
encoder_mf.scores["encoder"] == encoder_mf_count.scores["encoder"]

In [ ]:
encoder_mb_cond.verify()
encoder_mb_count.verify()
encoder_mf_count.verify()
encoder_mb.verify()

In [ ]:
from core.viz import plot_scatter

plot_scatter(
    x=encoder_mb.scores["encoder"],
    y=encoder_mf.scores["encoder"],
    xlabel="mb",
    ylabel="mf",
    add_unity=True,
    add_lr=True,
)
plot_scatter(
    x=encoder_mb_count.scores["encoder"],
    y=encoder_mf_count.scores["encoder"],
    xlabel="mb",
    ylabel="mf",
    add_unity=True,
    add_lr=True,
)

In [ ]:
# r2 between DMS and DLS
from core.viz import plot_kdes

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}
# scores
scores = {
    k: {
        f"{reg}, {model}": encoder_.scores[model][encoder_.reg_idxs[reg]]
        for reg in encoder_.regions
        for model in ["baseline", "encoder"]
    }
    for k, encoder_ in encoders.items()
}

# styles
linestyles = {"baseline": "--", "encoder": "-"}
colors = {"DMS": "#562E9C", "DLS": "#009D51"}
styles = {
    f"{reg}, {model}": {"linestyle": linestyles[model], "color": colors[reg]}
    for reg in encoder.regions
    for model in ["baseline", "encoder"]
}

for k, scores_ in scores.items():
    plot_kdes(
        scores_,
        label=rf"$r^2$, {k}",
        xlim=(-0.25, 1),
        add_means=False,
        line_kwargs=styles,
    )

In [ ]:
from core.data import colors_strategy

scores_baseline = {k: encoder_.scores["baseline"] for k, encoder_ in encoders.items()}
styles = {
    "full": {"color": "#444444", "linewidth": 1},
    "mb": {"color": colors_strategy["mb"], "linestyle": "--"},
    "mf": {"color": colors_strategy["mf"], "linestyle": "--"},
}
plot_kdes(scores_baseline, line_kwargs=styles)

## weight comp between regions and strategies

### kde

In [ ]:
# norm spike counts
from core.viz import plot_kde_row

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

sc_mean = {
    k: {
        reg: encoder_.robs[:, encoder_.reg_idxs[reg]].mean(axis=0)
        for reg in encoder_.regions
    }
    for k, encoder_ in encoders.items()
}

plot_kde_row(sc_mean)

In [ ]:
from core.viz import plot_kde_row
from core.data import colors_strategy
from utils.paths import FIGURES_DIR

encoders = {"full": encoder, "mb": encoder_mb, "mf": encoder_mf}

# styles
colors_region = {"DMS": "#2383DC", "DLS": "#3DC1E2"}
styles_strategy = {f"{reg}": {"color": colors_region[reg]} for reg in encoder.regions}
styles_reg = {f"{k}": {"color": colors_strategy[k]} for k in encoders}

# iterate through all regressors and save
for regressor in encoder.dm_names:
    if "tents" not in regressor:
        # weights
        weights_strategy = {
            k: {
                f"{reg}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for reg in encoder_.regions
            }
            for k, encoder_ in encoders.items()
        }

        weights_reg = {
            reg: {
                f"{k}": encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for k, encoder_ in encoders.items()
            }
            for reg in encoder.regions
        }

        # plot
        for k, weights, styles in zip(
            ["strategy", "region"],
            [weights_strategy, weights_reg],
            [styles_strategy, styles_reg],
        ):
            fig, _ = plot_kde_row(
                weights, styles, title=rf"$\beta$ {regressor}", add_means=False
            )

            fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight"
            fpath.mkdir(parents=True, exist_ok=True)
            fig.savefig(fpath / f"{regressor}_{k}.png", dpi=300, bbox_inches="tight")

            plt.close(fig)

### scatter, hist 2d, contour

In [ ]:
import numpy as np
from core.data import tv_vals
from core.viz import plot_scatter, plot_hist2d, plot_contour, plot_2d_row
from utils.viz_utils import save_fig

fpath = FIGURES_DIR / "lite" / subj_id / sess_id / "bweight"

for regr in encoder.tv_keys:
    vals = [tv_vals[regr][0]] if not regr == "response_prev" else tv_vals[regr]

    for val in vals:
        regressor = f"{regr}_{val}"
        weights = {
            reg: [
                encoder_.encoder_weights[
                    encoder_.reg_idxs[reg], encoder_.dm_idxs[regressor]
                ]
                for encoder_ in [encoder_mb, encoder_mf]
            ]
            for reg in encoder.regions
        }

        # scatter
        fig, ax = plot_2d_row(
            plot_scatter,
            weights,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            add_unity=True,
            add_lr=True,
        )

        save_fig(fig, fpath / "scatter", f"{regressor}.png")

        # mn/mx
        mn = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).min()
        )
        mx = (
            1.05
            * np.concatenate(
                [
                    weights_
                    for weights_strategy in weights.values()
                    for weights_ in weights_strategy
                ]
            ).max()
        )

        # hist2d
        fig, ax = plot_2d_row(
            plot_hist2d,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            sharey=True,
            sharex=True,
        )

        save_fig(fig, fpath / "hist2d", f"{regressor}.png")

        # contour
        fig, ax = plot_2d_row(
            plot_contour,
            weights,
            mn=mn,
            mx=mx,
            xlabel="mb",
            ylabel="mf",
            title=rf"$\beta$ {regressor}",
            fill=False,
            sharey=True,
            sharex=True,
        )

        save_fig(fig, fpath / "contour", f"{regressor}.png")